# Proposed validation — review before it runs

**What this measures:** The target is SRSNet's mean held-out MSE divided by PatchTST's over the repo's long-horizon grid (ETTh1/ETTh2/ETTm1/ETTm2/Weather × h=96/192/336/720), with both models trained and scored inside ONE run under identical input_size=512/loss/steps/seed — this answers the maintainers' "accuracy-parity vs PatchTST" with a measured baseline because SRSNet does not exist on main (capability, one arm, no sentinel; the run crashes non-zero if `neuralforecast.models.SRSNet` is unimportable). The script lives at `experiments/srsnet/run_srsnet.py` — the config-directory slot the repo's rules require a new method to add, in the `run_nhits.py` shape — loads datasets through the repo harness's own long-horizon loader (pinned group names, cached), honors `REMYX_SMOKE=1` by shrinking to ETTh1/h=96/a few steps with the same JSON keys, and writes its results beside it as the sibling experiments do.

**Target metric:** `srsnet_patchtst_mse_ratio`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

This notebook is the only copy of the test: `.remyx/validation.yaml` points here, and the run executes these cells top to bottom.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

## Execution context

The cells below are the test Remyx drafted, split into cells; it was written as a script, so `__file__` is set to `experiments/srsnet/run_srsnet.py` (no such file is committed). This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "experiments/srsnet/run_srsnet.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
"""
experiments/srsnet/run_srsnet.py — SRSNet vs PatchTST accuracy parity on the
repo long-horizon protocol (experiments/long_horizon shape): ETTh1/ETTh2/
ETTm1/ETTm2/Weather at h in {96,192,336,720}, MSE/MAE via utilsforecast on
rolling non-overlapping held-out tail windows. Both models are trained and
scored INSIDE this single run under identical lookback (input_size=512),
loss (MSE), steps, seed and splits — the parity baseline is measured, not
quoted. `neuralforecast.models.srsnet` is the PR's own module, imported
defensively for exactly the SRSNet symbol: on an arm where the change is
absent the SRSNet fits are skipped, every SRSNet-dependent metric degrades
to the 1e9 failure value with eval_cells=0 (fails the guardrail; degraded,
never fake parity) while PatchTST is still measured. Any data-loading or
training failure crashes with its traceback.

REMYX_SMOKE=1 -> ETTh1/h=96 only, 5 steps, 2 test windows, same JSON keys.
"""
import argparse
import json
import math
import os
import sys
import time

In [ ]:
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import torch
from datasetsforecast.long_horizon import LongHorizon
from utilsforecast.losses import mae, mse

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MSE
from neuralforecast.models import PatchTST

try:
    # module + symbol added by this PR; guarded so a baseline arm (change
    # absent) still runs and emits degraded metrics instead of crashing here
    from neuralforecast.models.srsnet import SRSNet
except (ImportError, AttributeError):
    SRSNet = None

In [ ]:
# Canonical long-horizon borders (Autoformer/PatchTST splits): keep = rows used
# per series (train+val+test tail), test = held-out tail scored by rolling
# windows; ETT hourly tail beyond border2 is unused by the protocol, ETTm and
# Weather run to series end.
GROUPS = {
    "ETTh1": {"freq": "h", "keep": 14400, "test": 2880},
    "ETTh2": {"freq": "h", "keep": 14400, "test": 2880},
    "ETTm1": {"freq": "15min", "keep": 57600, "test": 11520},
    "ETTm2": {"freq": "15min", "keep": 57600, "test": 11520},
    "Weather": {"freq": "h", "keep": None, "test": 10539},
}
HORIZONS = [96, 192, 336, 720]
INPUT_SIZE = 512  # patch-papers' standard lookback, pinned equal for both arms
STEPS, SEED = 3000, 1  # no early stopping: fixed budget, no Auto search on either arm
MAX_WINDOWS = 10 ** 6

In [ ]:
if os.environ.get("REMYX_SMOKE") == "1":
    GROUPS, HORIZONS, STEPS, MAX_WINDOWS = {"ETTh1": GROUPS["ETTh1"]}, [96], 5, 2

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "data")

def make_srsnet(h):
    # docs defaults from the PR's own docs/models.srsnet.html.md usage example
    return SRSNet(
        h=h, input_size=INPUT_SIZE, patch_len=16, stride=8, d_model=64,
        hidden_size=128, dropout=0.2, alpha=2.0, pos=True, loss=MSE(),
        max_steps=STEPS, random_seed=SEED, scaler_type=None,
        early_stop_patience_steps=-1,
    )

In [ ]:
def make_patchtst(h):
    # library defaults for everything the SRS module does not define
    return PatchTST(
        h=h, input_size=INPUT_SIZE, loss=MSE(), max_steps=STEPS,
        random_seed=SEED, scaler_type=None, early_stop_patience_steps=-1,
    )

In [ ]:
def alpha_gap(model):
    # Diagnostic: |sigmoid(alpha) - sigmoid(2.0)| after training (sigmoid(2.0)=0.881
    # is the SRS fusion-weight init; 0 means the SRS view never engaged). -1.0 if
    # the port registers no alpha-named parameter.
    ws = [p.flatten() for n, p in model.named_parameters() if "alpha" in n.lower()]
    if not ws:
        return -1.0
    w = torch.sigmoid(torch.cat(ws)).detach().cpu()
    return float((w - torch.sigmoid(torch.tensor(2.0))).abs().mean())

In [ ]:
def main():
    os.makedirs(DATA_DIR, exist_ok=True)
    cells, n_finite, n_pred, gaps = [], 0, 0, []
    for group, cfg in GROUPS.items():
        Y_df, _, _ = LongHorizon.load(directory=DATA_DIR, group=group)
        if Y_df["ds"].dtype == object:
            Y_df["ds"] = Y_df["ds"].astype("int64")
        if cfg["keep"] is not None:
            Y_df = Y_df.groupby("unique_id").head(cfg["keep"])
        for h in HORIZONS:
            n_windows = min(cfg["test"] // h, MAX_WINDOWS)
            rec = {"group": group, "h": h}
            arms = [("SRSNet", make_srsnet), ("PatchTST", make_patchtst)]
            if SRSNet is None:  # baseline arm: PR module absent -> degrade, never fake
                arms = [("PatchTST", make_patchtst)]
            for name, factory in arms:
                model = factory(h)
                nf = NeuralForecast(models=[model], freq=cfg["freq"])
                t0 = time.perf_counter()
                cv = nf.cross_validation(
                    df=Y_df, val_size=0, n_windows=n_windows, step_size=h
                )
                rec[f"{name.lower()}_sec"] = time.perf_counter() - t0
                cv = cv.drop(columns=["y"], errors="ignore").merge(
                    Y_df[["unique_id", "ds", "y"]], on=["unique_id", "ds"], how="inner"
                )
                preds = cv[name].to_numpy(dtype=float)
                rec[f"{name.lower()}_mse"] = float(mse(cv, models=[name])[name].mean())
                rec[f"{name.lower()}_mae"] = float(mae(cv, models=[name])[name].mean())
                if name == "SRSNet":
                    n_finite += int(np.isfinite(preds).sum())
                    n_pred += preds.size
                    gaps.append(alpha_gap(model))
            if SRSNet is None:  # NaN drops the cell from `ok` below -> eval_cells=0
                rec.update(srsnet_mse=float("nan"), srsnet_mae=float("nan"),
                           srsnet_sec=0.0)
            print(f"[cell] {group} h={h} srsnet_mse={rec['srsnet_mse']:.4f} "
                  f"patchtst_mse={rec['patchtst_mse']:.4f}", flush=True)
            cells.append(rec)
    ok = [c for c in cells if c["patchtst_mse"] > 0 and
          all(math.isfinite(c[k]) for k in ("srsnet_mse", "patchtst_mse", "srsnet_mae", "patchtst_mae"))]
    # 1e9 only reachable when eval_cells == 0 (which already fails that
    # guardrail) — e.g. SRSNet absent on this arm; it signals failure, never
    # fake parity.
    m = lambda xs: float(np.mean(xs)) if xs else 1e9
    t_srs = sum(c["srsnet_sec"] for c in cells)
    t_pth = sum(c["patchtst_sec"] for c in cells)
    metrics = {
        "srsnet_patchtst_mse_ratio": m([c["srsnet_mse"] / c["patchtst_mse"] for c in ok]),
        "srsnet_finite_pred_rate": (n_finite / n_pred) if n_pred else 0.0,
        "eval_cells": len(ok),
        "srsnet_mse": m([c["srsnet_mse"] for c in ok]),
        "patchtst_mse": m([c["patchtst_mse"] for c in ok]),
        "srsnet_mae": m([c["srsnet_mae"] for c in ok]),
        "patchtst_mae": m([c["patchtst_mae"] for c in ok]),
        "srsnet_patchtst_mae_ratio": m([c["srsnet_mae"] / c["patchtst_mae"] for c in ok]),
        "srs_alpha_gap": m([g for g in gaps if g >= 0]) if any(g >= 0 for g in gaps) else -1.0,
        # t_srs == 0 <=> SRSNet never ran on this arm: no measurable ratio
        "srsnet_patchtst_fit_time_ratio": (t_srs / t_pth) if (t_srs > 0 and t_pth > 0) else 1e9,
    }
    print(json.dumps({k: (round(v, 6) if isinstance(v, float) else v)
                      for k, v in metrics.items()}))

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--variant"); p.add_argument("--ref"); p.add_argument("--seed", type=int)
    p.parse_args()  # accepted and ignored: seed is pinned inside the protocol
    main()

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "maintainer @smellslikm: 'validate on GPU with the accuracy-parity vs PatchTST' — does SRSNet/PatchTST mean test-MSE ratio stay <= 1.0 on ETTh1/ETTh2/ETTm1/ETTm2/Weather at h=96/192/336/720?"
baseline: none
benchmarks:
  - name: srsnet-patchtst-long-horizon
    suite: "experiments/srsnet/run_srsnet.py"
    packages: ["datasetsforecast"]
    scorer: srsnet_patchtst_mse_ratio
    baseline: none
    policy:
      guardrail_veto: true
    metrics:
      - name: srsnet_patchtst_mse_ratio
        direction: min
        threshold: 1.0
        role: target
        bar: goal
        reads_as: "SRSNet MSE / PatchTST MSE averaged over the 20 dataset-horizon cells; 1.0 is exact parity, below 1.0 backs the paper's stronger claim"
      - name: srsnet_finite_pred_rate
        direction: max
        threshold: 1.0
        role: guardrail
        bar: sanity
        reads_as: "Share of SRSNet held-out forecasts that are all-finite; any NaN/Inf fails it (the PR's own unit test asserts finiteness)"
      - name: eval_cells
        direction: max
        threshold: 20
        role: guardrail
        bar: sanity
        reads_as: "Dataset x horizon cells actually measured, out of the 20 the protocol defines (5 datasets x 4 horizons)"
      - name: srsnet_mse
        direction: min
        threshold: 5.0
        role: diagnostic
        bar: sanity
        reads_as: "SRSNet mean held-out MSE over the grid, the repo protocol's own metric on standardized targets (ETT/weather cells run ~0.15-1.0)"
      - name: patchtst_mse
        direction: min
        threshold: 5.0
        role: diagnostic
        bar: sanity
        reads_as: "Measured PatchTST arm's mean held-out MSE on the identical windows; this is the parity denominator"
      - name: srsnet_mae
        direction: min
        threshold: 5.0
        role: diagnostic
        bar: sanity
        reads_as: "SRSNet mean held-out MAE over the grid, second paper metric"
      - name: patchtst_mae
        direction: min
        threshold: 5.0
        role: diagnostic
        bar: sanity
        reads_as: "Measured PatchTST arm's mean held-out MAE on the identical windows"
      - name: srsnet_patchtst_mae_ratio
        direction: min
        threshold: 2.0
        role: diagnostic
        bar: sanity
        reads_as: "SRSNet MAE / PatchTST MAE; shows whether MAE tracks the MSE verdict"
      - name: srs_alpha_gap
        direction: max
        threshold: 1.0
        role: diagnostic
        bar: sanity
        reads_as: "Mean |sigmoid(alpha) - 0.881| after training; 0 means the fusion weight never moved and the SRS view is inert"
      - name: srsnet_patchtst_fit_time_ratio
        direction: min
        role: cost
        reads_as: "SRSNet fit seconds / PatchTST fit seconds averaged over cells; the SRS scorers' compute overhead (~5M MACs per channel-window), cost context only"
compute:
  tier: gpu
  # one arm = 40 fits (2 models x 20 cells) at max_steps=3000, windows_batch_size 256:
  # ~2-4 min/fit on one GPU + pinned dataset download + rolling test cross-validation
  # ~= 3 h; 14400 s leaves honest margin. REMYX_SMOKE=1 shrinks to ETTh1/h=96/few steps.
  timeout_s: 14400
loop:
  max_iterations: 8
  fix_code: true
held_constant:
  - "input_size=512 for both models (patch-papers' standard lookback; pinned equal so the arms differ only by the SRS module)"
  - "loss=MSE, max_steps=3000, no early stopping, library batch/windows defaults, random_seed=1 for both models"
  - "train/test splits and rolling non-overlapping test windows from the repo long-horizon loader groups ETTh1/ETTh2/ETTm1/ETTm2/Weather at h in {96,192,336,720}"
  - "SRSNet at docs defaults (patch_len=16, stride=8, d_model=64, hidden_size=128, dropout=0.2, alpha=2.0, pos=True, revin=True) vs PatchTST at its library defaults; scaler_type=None on both"
  - "MSE/MAE computed with utilsforecast.losses on the same held-out tail windows"
avoid:
  - "no AutoSRSNet/AutoPatchTST search on either arm — tuned-vs-fixed would confound the parity question"
  - "no quoted PatchTST number as baseline — the repo has no canonical row (maturity R2); the baseline is measured inside the run"
  - "no RevIN + scaler_type double normalization — scaler_type stays None on both arms"
  - "no fake parity for the absent model — the PR-added module is imported defensively (ImportError/AttributeError caught for exactly the SRSNet symbol): on an arm without the change the SRSNet fits are skipped, its metrics degrade to the 1e9 failure value and eval_cells=0 fails the guardrail while PatchTST is still measured; dataset-load or training failures still crash with tracebacks rather than printing zeroed metrics"
  - "no stand-in datasets — if the Weather group is absent from the repo's own loader the run crashes rather than substituting another dataset"
  - "no wall-clock gating — the fit-time ratio is cost context on the dedicated GPU, never a threshold"
provenance:
  srsnet_patchtst_mse_ratio: "maintainer_comment:@smellslikm (accuracy-parity vs PatchTST on GPU)"
  srsnet_mse: "protocol_doc:experiments/long_horizon/README.md (MSE per dataset per horizon)"
  patchtst_mse: "protocol_doc:experiments/long_horizon/README.md"
  srsnet_mae: "protocol_doc:experiments/long_horizon/README.md (MAE per dataset per horizon)"
  patchtst_mae: "protocol_doc:experiments/long_horizon/README.md"
  srsnet_finite_pred_rate: "pr_diff:tests/test_models/test_srsnet.py (asserts forecast finiteness)"
  srs_alpha_gap: "pr_diff:neuralforecast/models/srsnet.py (alpha default 2.0, sigmoid(2.0)=0.881)"
  eval_cells: "user_guidance (ETT+Weather at 96/192/336/720)"
  held_constant: "protocol_doc:experiments/long_horizon/README.md"
  suite: "synthesized (mirrors experiments/long_horizon/run_nhits.py shape, placed in the experiments/srsnet slot repo rules require for a new method)"
report:
  headline: "SRSNet trains and forecasts end-to-end on the repo's long-horizon protocol; parity vs the measured PatchTST arm is read off the MSE ratio against 1.0"
  findings:
    - "SRSNet/PatchTST mean test-MSE ratio = {srsnet_patchtst_mse_ratio} (<=1.0 parity, <1.0 supports the paper's SOTA direction)"
    - "Absolute means: SRSNet MSE {srsnet_mse} vs PatchTST {patchtst_mse}; MAE {srsnet_mae} vs {patchtst_mae} (MAE ratio {srsnet_patchtst_mae_ratio})"
    - "{srsnet_finite_pred_rate:%} of SRSNet held-out forecasts are finite"
    - "{eval_cells} of 20 dataset-horizon cells measured"
    - "Fusion weight moved |sigmoid(alpha) - 0.881| = {srs_alpha_gap} from init (0 means the SRS view never engaged)"
    - "SRSNet fit time = {srsnet_patchtst_fit_time_ratio}x PatchTST's (scorer overhead; cost context only)"
  establishes:
    - "that the ported SRSNet trains under neuralforecast's BaseModel on the repo's real long-horizon datasets and reaches mean MSE at or below a PatchTST trained under identical lookback/loss/steps/seed"
    - "that the SRS fusion weight trains away from its 0.881 init, i.e. the new module is engaged rather than inert"
  does_not_establish:
    - "the paper's SOTA ranking (its numbers come from the TFB/ts_benchmark harness with its own LR schedule, epochs and fixed RevIN; only the ETT+Weather subset is testable here)"
    - "the plug-and-play claim that SRS improves other patch models (that module path is not ported)"
    - "any latency, throughput or memory advantage (fit-time ratio is context, not a benchmark)"
  not_measured:
    - "tuned comparison AutoSRSNet vs AutoPatchTST with equal search budgets"
    - "datasets beyond ETTh1/ETTh2/ETTm1/ETTm2/Weather from the paper's multi-domain suite"
    - "parameter-count parity (the scorers add ~20.6k params per SRS layer — bookkeeping only, never gated)"
  caveat: "Same architecture does not transfer the published row: the port trains under neuralforecast BaseModel (windows_batch_size sampling, no TFB LR schedule), so the PatchTST arm is measured under the identical recipe rather than quoted."
  next: "If the ratio clears 1.0 at fixed defaults, rerun with equal Auto-search budgets (AutoSRSNet vs AutoPatchTST) on the same grid to test the stronger tuned claim."
```